# Day 23: Tracing & Observability

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> If you can't see what your agent is doing, you can't fix it when it breaks.

Tracing is how you debug, monitor, and improve agent systems. It shows you
every LLM call, tool invocation, and decision point. Today we explore what
each framework offers out of the box and how to add custom tracing.

## What You'll Build
- Custom tracing wrappers for all 3 frameworks
- Token usage tracking
- Timing measurements
- A trace comparison table

In [10]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 23 SETUP COMPLETE")
print("=" * 50)
print_config()

# Shared inputs across all three framework cells (defined here so each cell is self-contained)
questions = [
    "What is 25 * 47?",
    "What is 144 divided by 12?",
]

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 23 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


In [11]:
# ── Custom Tracer (works with all frameworks) ─────────────
import time

class AgentTracer:
    """Simple tracing for agent calls — no external dependencies."""

    def __init__(self, name: str):
        self.name = name
        self.steps = []

    def start(self, label: str):
        step = {"label": label, "start_time": time.time()}
        self.steps.append(step)
        return step

    def end(self, label: str, tokens: dict | None = None, **meta):
        now = time.time()
        for step in reversed(self.steps):
            if step["label"] == label and "end_time" not in step:
                step["end_time"] = now
                step["duration_ms"] = round((now - step["start_time"]) * 1000)
                step["tokens"] = tokens or {}
                step.update(meta)
                break

    def report(self):
        print(f"\n{'═' * 60}")
        print(f"  TRACE: {self.name}")
        print(f"{'═' * 60}")
        total_ms = 0
        total_in = 0
        total_out = 0
        for i, step in enumerate(self.steps, 1):
            duration = step.get("duration_ms", "?")
            if isinstance(duration, (int, float)):
                total_ms += duration
            tok = step.get("tokens") or {}
            tok_str = ""
            if tok:
                tok_str = f" | in: {tok.get('input', 0)}  out: {tok.get('output', 0)}"
                total_in += tok.get("input", 0) or 0
                total_out += tok.get("output", 0) or 0
            else:
                tok_str = " | tokens: n/a"
            print(f"  {i}. {step['label']:<30} {duration:>6} ms{tok_str}")
        print(f"{'─' * 60}")
        print(f"  TOTAL: {total_ms} ms | {len(self.steps)} steps | "
              f"tokens in: {total_in}  out: {total_out}")
        print(f"{'═' * 60}\n")
        return {
            "total_ms": total_ms,
            "steps": len(self.steps),
            "tokens_in": total_in,
            "tokens_out": total_out,
        }

print("Custom tracer ready — no external observability tools needed.")
print("In production, you'd use: LangSmith, Phoenix, or OpenTelemetry.")

Custom tracer ready — no external observability tools needed.
In production, you'd use: LangSmith, Phoenix, or OpenTelemetry.


---
## Framework 1: OpenAI Agents SDK — Tracing

In [12]:
from agents import Agent, Runner, function_tool, set_tracing_disabled
from openai import AsyncOpenAI
from shared import OLLAMA_MODEL, OLLAMA_BASE_URL

# Re-enable the SDK's built-in observability for this cell (tested: flag alone
# does not fix the 0/0 — the cause is below).
set_tracing_disabled(False)

model = get_openai_agents_model()
tracer = AgentTracer("OpenAI SDK")

# A plain non-streaming client used only for the usage fallback below.
diag_client = AsyncOpenAI(api_key="ollama", base_url=f"{OLLAMA_BASE_URL}/v1")

@function_tool
def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        result = eval(expression)  # Only for learning!
        return str(result)
    except Exception as e:
        return f"Error: {e}"

agent = Agent(
    name="Math Agent",
    instructions="You solve math problems. Use the calculator tool. Show your work.",
    tools=[calculate],
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — TRACING (with usage fallback)")
print("=" * 60)

for q in questions:
    print(f"\n>>> {q}")
    tracer.start(f"run: {q[:30]}")
    result = await Runner.run(agent, q, max_turns=5)

    # Read usage the SDK aggregated
    usage = getattr(result, "usage", None) or {}
    tokens = {
        "input": getattr(usage, "input_tokens", None) or (usage.get("input_tokens", 0) if isinstance(usage, dict) else 0),
        "output": getattr(usage, "output_tokens", None) or (usage.get("output_tokens", 0) if isinstance(usage, dict) else 0),
    }
    source = "result.usage"

    # FALLBACK: the SDK streams internally without stream_options.include_usage,
    # so Ollama never emits a usage chunk and result.usage aggregates to 0/0
    # (verified outside the notebook: stream=True -> usage None; stream=False ->
    # usage present). When the SDK hides the number, replay the prompt
    # non-streamed against the same endpoint and read the real usage.
    # Note: this counts the user prompt only — the agent's true input also
    # includes the system prompt + tool defs the SDK injects, so this is a floor.
    if tokens["input"] == 0 and tokens["output"] == 0:
        replay = await diag_client.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": q}],
            stream=False,
        )
        tokens = {
            "input": replay.usage.prompt_tokens,
            "output": replay.usage.completion_tokens,
        }
        source = "fallback (non-streamed replay)"

    tracer.end(f"run: {q[:30]}", tokens=tokens, source=source)
    print(f"ANSWER: {result.final_output[:200]}")
    print(f"   tokens {tokens['input']} in / {tokens['output']} out  [{source}]")

openai_trace = tracer.report()

print("\nRoot cause (verified outside the notebook via the openai client):")
print("  stream=False                    -> usage present (prompt_tokens, completion_tokens)")
print("  stream=True, no stream_options  -> usage None   <- the SDK hits this path")
print("  stream=True, include_usage      -> usage present")
print("Ollama returns usage correctly. The SDK streams without include_usage,")
print("so the usage chunk never arrives and result.usage aggregates to 0/0.")
print("The fallback above replays the prompt non-streamed to recover the number.")

# Restore the global setting so downstream cells stay quiet against Ollama
set_tracing_disabled(True)

OPENAI AGENTS SDK — TRACING (with usage fallback)

>>> What is 25 * 47?


ANSWER: The result of 25 multiplied by 47 is 1175.
   tokens 39 in / 18 out  [fallback (non-streamed replay)]

>>> What is 144 divided by 12?


ANSWER: 144 divided by 12 is 12.0.
   tokens 41 in / 14 out  [fallback (non-streamed replay)]

════════════════════════════════════════════════════════════
  TRACE: OpenAI SDK
════════════════════════════════════════════════════════════
  1. run: What is 25 * 47?            8427 ms | in: 39  out: 18
  2. run: What is 144 divided by 12?   5096 ms | in: 41  out: 14
────────────────────────────────────────────────────────────
  TOTAL: 13523 ms | 2 steps | tokens in: 80  out: 32
════════════════════════════════════════════════════════════


Root cause (verified outside the notebook via the openai client):
  stream=False                    -> usage present (prompt_tokens, completion_tokens)
  stream=True, no stream_options  -> usage None   <- the SDK hits this path
  stream=True, include_usage      -> usage present
Ollama returns usage correctly. The SDK streams without include_usage,
so the usage chunk never arrives and result.usage aggregates to 0/0.
The fallback above replays the prompt 

---
## Framework 2: LangGraph — Tracing

In [4]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from typing_extensions import TypedDict, Annotated
import operator

model = get_langgraph_model()
tracer = AgentTracer("LangGraph")

@tool
def calculate_lg(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

class MathState(TypedDict):
    messages: Annotated[list, operator.add]
    query: str
    answer: str

def solve_node(state: MathState) -> dict:
    r = model.invoke([
        SystemMessage(content="Solve the math problem step by step."),
        HumanMessage(content=state["query"]),
    ])
    return {"messages": [r], "answer": r.content}

builder = StateGraph(MathState)
builder.add_node("solve", solve_node)
builder.add_edge(START, "solve")
builder.add_edge("solve", END)
graph = builder.compile()

print("=" * 60)
print("LANGGRAPH — TRACING")
print("=" * 60)

for q in questions:
    print(f"\n>>> {q}")
    tracer.start(f"invoke: {q[:30]}")
    result = graph.invoke({"messages": [], "query": q, "answer": ""})
    # Real usage from AIMessage.usage_metadata (populated by ChatOllama / ChatOpenAI)
    msg = result["messages"][-1]
    um = getattr(msg, "usage_metadata", None) or {}
    tokens = {
        "input": um.get("input_tokens", 0) if isinstance(um, dict) else getattr(um, "input_tokens", 0),
        "output": um.get("output_tokens", 0) if isinstance(um, dict) else getattr(um, "output_tokens", 0),
    }
    tracer.end(f"invoke: {q[:30]}", tokens=tokens)
    print(f"ANSWER: {result['answer'][:200]}")

langgraph_trace = tracer.report()

print("LangGraph tracing options:")
print("  - LangSmith: full trace visualization (free tier available)")
print("  - Callback handlers: on_llm_start, on_tool_start, etc.")
print("  - graph.get_graph().draw_mermaid() for graph visualization")
print("  - State inspection at every node")
print("  - Token counts above come from AIMessage.usage_metadata")

LANGGRAPH — TRACING

>>> What is 25 * 47?
ANSWER: To solve this multiplication problem, we can use the standard algorithm for multiplying two numbers.

Step 1: Multiply 5 (the units digit of 25) by 47.
\(5 \times 47 = 235\)

Step 2: Multiply 20 (the 

>>> What is 144 divided by 12?
ANSWER: To solve this division problem, we can break it down into simple steps:

Step 1: Understand that dividing 144 by 12 means finding out how many groups of 12 can be made from the total of 144.

Step 2: 

════════════════════════════════════════════════════════════
  TRACE: LangGraph
════════════════════════════════════════════════════════════
  1. invoke: What is 25 * 47?        25672 ms | in: 32  out: 351
  2. invoke: What is 144 divided by 12?  16596 ms | in: 34  out: 231
────────────────────────────────────────────────────────────
  TOTAL: 42268 ms | 2 steps | tokens in: 66  out: 582
════════════════════════════════════════════════════════════

LangGraph tracing options:
  - LangSmith: full trace vi

---
## Framework 3: CrewAI — Tracing

In [16]:
from crewai import Agent, Task, Crew, Process
import litellm

llm = get_crewai_llm()
tracer = AgentTracer("CrewAI")

# CrewOutput has no usage field. But CrewAI runs every completion through LiteLLM,
# and LiteLLM fires a success callback with the real usage on every call.
# Tap that layer directly. Same "go under the abstraction" move as the OpenAI
# SDK cell's non-streamed replay, but cleaner by design: captures real usage
# from the actual crew run, no second call.
# Honest caveat from this run: in this CrewAI version the tap registered but
# did not fire. Cell output prints "LiteLLM tap did not fire" when that happens
# rather than faking a number. The number still exists at the LiteLLM layer.
_crew_usage_log = []

class _UsageTap(litellm.CustomLogger):
    def log_success(self, kwargs, response_obj, start_time, end_time):
        u = getattr(response_obj, "usage", None)
        if u is None:
            return
        _crew_usage_log.append({
            "input": getattr(u, "prompt_tokens", 0) or 0,
            "output": getattr(u, "completion_tokens", 0) or 0,
        })

_usage_tap = _UsageTap()
litellm.success_callback = [_usage_tap]

math_agent = Agent(
    role="Math Expert",
    goal="Solve math problems accurately",
    backstory="You are a mathematician who shows step-by-step work.",
    llm=llm, verbose=True,
)

print("=" * 60)
print("CREWAI — TRACING (usage tapped at the LiteLLM layer)")
print("=" * 60)

for q in questions:
    print(f"\n>>> {q}")

    task = Task(
        description=f"Solve this math problem: {q}",
        expected_output="The numerical answer with steps shown",
        agent=math_agent,
    )

    _crew_usage_log.clear()
    tracer.start(f"kickoff: {q[:30]}")
    crew = Crew(agents=[math_agent], tasks=[task], process=Process.sequential, verbose=True)
    result = await crew.kickoff_async()

    # Aggregate usage across every LiteLLM call this crew made (the agent may
    # do more than one completion per task).
    if _crew_usage_log:
        tokens = {
            "input": sum(x["input"] for x in _crew_usage_log),
            "output": sum(x["output"] for x in _crew_usage_log),
        }
        source = f"LiteLLM tap ({len(_crew_usage_log)} call(s))"
    else:
        tokens = None
        source = "LiteLLM tap did not fire"

    tracer.end(f"kickoff: {q[:30]}", tokens=tokens, source=source)
    print(f"ANSWER: {str(result)[:200]}")
    if tokens:
        print(f"   tokens {tokens['input']} in / {tokens['output']} out  [{source}]")
    else:
        print(f"   {source}")

crewai_trace = tracer.report()

# Remove the global callback so it doesn't leak into other cells
litellm.success_callback = []

print("\nCrewAI tracing options:")
print("  - verbose=True (used above): step-by-step agent reasoning in console, noisy")
print("  - tracing=True on Crew: routes spans to CrewAI's cloud, needs an account")
print("  - LITELLM TAP (used above): CustomLogger on litellm.success_callback to capture")
print("    usage from the layer CrewAI runs on. Bypasses the missing CrewOutput.usage")
print("    field. In this run it registered but did not fire. CrewAI's call path in")
print("    this version didn't trigger the documented success callback.")
print("  - CrewAI Plus: paid telemetry tier")

CREWAI — TRACING (usage tapped at the LiteLLM layer)

>>> What is 25 * 47?


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.15.1                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ac149bbb-1011-4933-80c5-b7390f154814                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Solve this math problem: What is 25 * 47?                                                                │
│  ID: dd68a1df-cbf2-4059-b154-86fc33fd7f8f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Expert                                                                                             │
│                                                                                                                 │
│  Task: Solve this math problem: What is 25 * 47?                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Expert                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To solve the problem of dividing 144 by 12, we will go through each step systematically:                       │
│                                                                                                                 │
│  1. We start with the dividend which is 144 and the divisor which is 12.                                        │
│  2. Dividing large numbers like this can often be done mentally or remembered for quick answers.                │
│                                                                                                                 │
│  Let's proceed manually:                                                                                        │
│  3. Imagine breaking down 144 into groups of 12:                                                                │
│     - First grouping: We ask how many times 12 fits into 14 (the first two digits). It goes 1 time (since 12 x  │
│  1 = 12).                                                                                                       │
│     - Subtracting 12 from 14 gives a remainder of 2.                                                            │
│     - Now, consider the "24" part in 144. Since we've already considered it as a whole number with a leading    │
│  zero, imagine this as two groups and add their contributions directly:                                         │
│        - How many times does 12 fit into 24? It goes exactly 2 times (since 12 x 2 = 24).                       │
│                                                                                                                 │
│  4. Adding these results together:                                                                              │
│     - The contribution from the first grouping was 1.                                                           │
│     - And the second grouping also gave us a total of 2.                                                        │
│                                                                                                                 │
│  Therefore, when we combine both contributions, they add up to 1 + 2 = 3.                                       │
│                                                                                                                 │
│  So, \( \text{144} \div \text{12} = \text{12} \)                                                                │
│                                                                                                                 │
│  Final numerical answer: The number you get from dividing 144 by 12 is **12**.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Solve this math problem: What is 144 divided by 12?                                                      │
│  Agent: Math Expert                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 86f03fca-1848-4d19-a4d7-e2ee0ed71b84                                                                       │
│  Final Output: To solve the problem of dividing 144 by 12, we will go through each step systematically:         │
│                                                                                                                 │
│  1. We start with the dividend which is 144 and the divisor which is 12.                                        │
│  2. Dividing large numbers like this can often be done mentally or remembered for quick answers.                │
│                                                                                                                 │
│  Let's proceed manually:                                                                                        │
│  3. Imagine breaking down 144 into groups of 12:                                                                │
│     - First grouping: We ask how many times 12 fits into 14 (the first two digits). It goes 1 time (since 12 x  │
│  1 = 12).                                                                                                       │
│     - Subtracting 12 from 14 gives a remainder of 2.                                                            │
│     - Now, consider the "24" part in 144. Since we've already considered it as a whole number with a leading    │
│  zero, imagine this as two groups and add their contributions directly:                                         │
│        - How many times does 12 fit into 24? It goes exactly 2 times (since 12 x 2 = 24).                       │
│                                                                                                                 │
│  4. Adding these results together:                                                                              │
│     - The contribution from the first grouping was 1.                                                           │
│     - And the second grouping also gave us a total of 2.                                                        │
│                                                                                                                 │
│  Therefore, when we combine both contributions, they add up to 1 + 2 = 3.                                       │
│                                                                                                                 │
│  So, \( \text{144} \div \text{12} = \text{12} \)                                                                │
│                                                                                                                 │
│  Final numerical answer: The number you get from dividing 144 by 12 is **12**.                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ANSWER: To solve the problem of dividing 144 by 12, we will go through each step systematically:

1. We start with the dividend which is 144 and the divisor which is 12.
2. Dividing large numbers like this ca
   LiteLLM tap did not fire

════════════════════════════════════════════════════════════
  TRACE: CrewAI
════════════════════════════════════════════════════════════
  1. kickoff: What is 25 * 47?       41657 ms | tokens: n/a
  2. kickoff: What is 144 divided by 12?  23219 ms | tokens: n/a
────────────────────────────────────────────────────────────
  TOTAL: 64876 ms | 2 steps | tokens in: 0  out: 0
════════════════════════════════════════════════════════════


CrewAI tracing options:
  - verbose=True (used above): step-by-step agent reasoning in console, noisy
  - tracing=True on Crew: routes spans to CrewAI's cloud, needs an account
  - LITELLM TAP (used above): CustomLogger on litellm.success_callback to capture
    usage from the layer CrewAI runs on. Bypasses the missing Cre

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
# ── Measured comparison (single source of truth — re-runs always reflect this run) ──
print("=" * 64)
print("  MEASURED COMPARISON — this run")
print("=" * 64)
print(f"  {'Framework':<16} {'Time (ms)':>10} {'Tokens in':>11} {'Tokens out':>12}")
print("  " + "-" * 62)
for name, t in [("OpenAI SDK", openai_trace),
                ("LangGraph", langgraph_trace),
                ("CrewAI", crewai_trace)]:
    print(f"  {name:<16} {t['total_ms']:>10} {t['tokens_in']:>11} {t['tokens_out']:>12}")
print("  " + "-" * 62)

print("\nWhat this run established (read from the rows above, not from static text):")

if openai_trace["tokens_in"] or openai_trace["tokens_out"]:
    print(f"  - OpenAI Agents SDK: recovered {openai_trace['tokens_in']} in / "
          f"{openai_trace['tokens_out']} out via non-streamed replay fallback.")
    print(f"    result.usage was 0/0 (SDK streams without stream_options.include_usage).")
    print(f"    Floor: counts the user prompt only, not the system prompt + tool defs the agent sends.")
else:
    print(f"  - OpenAI Agents SDK: 0/0 (fallback did not recover usage this run).")

print(f"  - LangGraph: {langgraph_trace['tokens_in']} in / {langgraph_trace['tokens_out']} out "
      f"native, from AIMessage.usage_metadata. Full context counted.")

if crewai_trace["tokens_in"] or crewai_trace["tokens_out"]:
    print(f"  - CrewAI: recovered {crewai_trace['tokens_in']} in / {crewai_trace['tokens_out']} out via LiteLLM tap.")
else:
    print(f"  - CrewAI: LiteLLM tap wired (CustomLogger on litellm.success_callback) but did")
    print(f"    not fire in this CrewAI version. Usage still not surfaced. Open problem.")

print(f"  - Timing order stable across runs; exact ms varies. OpenAI SDK is the baseline;")
print(f"    LangGraph and CrewAI each ran several times longer on the same prompts and model.")

  MEASURED COMPARISON — this run
  Framework         Time (ms)   Tokens in   Tokens out
  --------------------------------------------------------------
  OpenAI SDK            13523          80           32
  LangGraph             42268          66          582
  CrewAI                64876           0            0
  --------------------------------------------------------------

What this run established (read from the rows above, not from static text):
  - OpenAI Agents SDK: recovered 80 in / 32 out via non-streamed replay fallback.
    result.usage was 0/0 (SDK streams without stream_options.include_usage).
    Floor: counts the user prompt only, not the system prompt + tool defs the agent sends.
  - LangGraph: 66 in / 582 out native, from AIMessage.usage_metadata. Full context counted.
  - CrewAI: LiteLLM tap wired (CustomLogger on litellm.success_callback) but did
    not fire in this CrewAI version. Usage still not surfaced. Open problem.
  - Timing order stable across runs; exa

---
## Observability Comparison

The row below describes the **method and outcome** per framework on the Ollama path. Run-specific numbers live in the MEASURED COMPARISON code cell directly above this one and always match the latest run.

| Feature | OpenAI SDK | LangGraph | CrewAI |
|---|---|---|---|
| **Built-in tracing** | OpenAI dashboard | LangSmith | Verbose mode |
| **Token tracking (this run)** | `result.usage` 0/0; recovered via non-streamed replay (a floor, user-prompt only) | `AIMessage.usage_metadata`, real numbers, native, full context | LiteLLM tap wired (CustomLogger on `success_callback`); did not fire in this CrewAI version |
| **Tool call visibility** | In trace | In trace | In verbose log |
| **Custom tracing** | Callbacks | Callbacks | Callbacks |
| **Latency (this run)** | Baseline | Several times longer | Several times longer |

### Production Observability Stack

```
Level 1: Console logging (timing always works; token counts exposed OR recovered per framework)
Level 2: Recovery by going under the abstraction (the move this notebook taught)
Level 3: Structured logging (JSON traces to file/database; replay)
Level 4: APM tools (LangSmith, Phoenix Arize, OpenTelemetry)
Level 5: Full observability (alerts, dashboards, anomaly detection)
```

### What this run established (see the MEASURED COMPARISON cell above for exact numbers)

- **LangGraph**: the only framework that exposed real token counts natively on the local path, via `AIMessage.usage_metadata`. No recovery needed. Full context counted.
- **OpenAI Agents SDK**: `result.usage` came back 0/0. Root cause, verified outside the notebook with three direct calls to the same Ollama endpoint: non-streamed returns `usage`, streamed without `stream_options={"include_usage": True}` returns `None`, streamed with the flag returns `usage`. The SDK streams without the flag, so Ollama never emits a usage chunk and the SDK aggregates to 0/0. Cell 4 recovers a real number with a non-streamed replay of the user prompt. Floor, not exact: the agent's full input also includes the system prompt and tool definitions the replay doesn't send.
- **CrewAI**: `CrewOutput` has no usage field, even with `tracing=True`. Cell 8 wires a `CustomLogger` on `litellm.success_callback` to tap the layer CrewAI runs on. The tap registered but **did not fire** in this CrewAI version. The number still exists at the LiteLLM layer; the documented callback path didn't trigger it. Open problem, not a solved one.
- **Timing**: OpenAI SDK is the baseline; LangGraph and CrewAI each ran several times longer on the same prompts and model. Order stable across runs, exact ms not.

The deeper lesson: feature checklists compare what frameworks claim. The matrix that matters is which abstraction hides the number, and how hard it is to go under it. One framework here still resisted.

## Key Takeaway

Tracing is essential for production agents. But "has tracing" is the marketing layer. The practitioner question is sharper: **when the abstraction hides the number, can you get it back?**

1. **Console logging** — timing always works; real token counts only where the framework exposes them or where you recover them yourself.
2. **Recovery by going under the abstraction** — the move this notebook actually taught. Three variants, three outcomes:
   - **LangGraph**: no recovery needed. `AIMessage.usage_metadata` is on every message. Read the state. Full context counted.
   - **OpenAI Agents SDK**: `result.usage` came back 0/0. Diagnosed why (the SDK streams internally without `stream_options={"include_usage": True}`, and Ollama only emits a usage chunk when asked). Recovered a real number with a non-streamed replay of the user prompt. Floor: the agent's full input also includes the system prompt and tool defs the replay doesn't send.
   - **CrewAI**: `CrewOutput` has no usage field. Wired a `CustomLogger` on `litellm.success_callback` to tap the layer CrewAI runs on. The tap registered but did not fire in this CrewAI version. The number still exists at the LiteLLM layer; the documented callback path didn't trigger it. Open problem, not a solved one.
3. **Dedicated tools** — LangSmith, OpenAI dashboard, Phoenix Arize, OpenTelemetry. The marketing layer's answer; useful when it works, none of them surface Ollama-path tokens out of the box here.

The lesson: feature checklists compare what frameworks claim. The matrix that matters is which abstraction hides the number, how hard it is to go under it, and which ones still resist. One framework here still resisted, and that is the honest result.

---

